# 🔍 05 — Explainable AI (SHAP & LIME)

**CreditLens AI — Capstone Project**

This notebook demonstrates **Explainable AI (XAI)** — the differentiating factor of this project. We don't just predict; we **explain why** a loan was approved or rejected.

1. **SHAP Analysis**
   - Summary Plot (beeswarm) — global feature impact
   - Feature Importance bar chart — mean |SHAP value|
   - Waterfall plots — per-applicant decision breakdown
   - Force plots — prediction pushes from base value
2. **LIME Analysis**
   - Local explanations — feature contribution tables
   - Explanation plots — horizontal bar charts
   - What-If Analysis — change features and observe impact
3. **SHAP vs LIME Comparison** — side-by-side for same applicant

---

In [ ]:
# Standard imports
import sys
import json
from pathlib import Path

# Add project root to path
PROJECT_ROOT = Path.cwd().parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import shap

# Project imports
from src.config import REPORTS_DIR, MODELS_DIR, TARGET_COLUMN

# Plotting config
sns.set_theme(style="whitegrid", palette="viridis")
plt.rcParams.update({"figure.figsize": (12, 6), "figure.dpi": 120})

# Initialise SHAP JS for interactive plots
shap.initjs()

print(f"Project root: {PROJECT_ROOT}")
print(f"SHAP version: {shap.__version__}")

## 1. Load Model & Prepare Data

We load the best trained model and prepare a subset of test data for explanation generation.

In [ ]:
import joblib

from src.data.merger import load_and_merge_datasets
from src.data.loader import validate_dataset, split_data
from src.features.engineer import engineer_features
from src.data.preprocessor import preprocess_data

# Load & preprocess data
df = load_and_merge_datasets()
df = validate_dataset(df)
df = engineer_features(df)
X_train, X_test, y_train, y_test = split_data(df)
X_train_processed, X_test_processed, y_train_resampled, _, feature_names = preprocess_data(
    X_train, X_test, y_train, apply_smote=True
)

# Load the best model
model = joblib.load(MODELS_DIR / "best_model.joblib")
print(f"Model loaded: {type(model).__name__}")
print(f"Test set shape: {X_test_processed.shape}")
print(f"Feature count: {len(feature_names)}")
print(f"Features: {feature_names}")

In [ ]:
# Get predictions for the test set
y_pred = model.predict(X_test_processed)
y_proba = model.predict_proba(X_test_processed)[:, 1]

# Find sample indices for approved and rejected predictions
approved_indices = np.where(y_pred == 1)[0]
rejected_indices = np.where(y_pred == 0)[0]

print(f"Predictions on test set:")
print(f"  Approved: {len(approved_indices):,}")
print(f"  Rejected: {len(rejected_indices):,}")

# Select sample applicants for explanation
sample_approved = approved_indices[0]
sample_rejected = rejected_indices[0]

print(f"\nSample approved applicant (index {sample_approved}): P(approval) = {y_proba[sample_approved]:.4f}")
print(f"Sample rejected applicant (index {sample_rejected}): P(approval) = {y_proba[sample_rejected]:.4f}")

---

## 2. SHAP Analysis

**SHAP (SHapley Additive exPlanations)** uses game-theory Shapley values to attribute each feature's contribution to the prediction. It answers: *"How much did each feature push the prediction toward approval or rejection?"*

### 2.1 Initialise SHAP Explainer

In [ ]:
from src.explainability.shap_explainer import SHAPExplainer

# Use a subset of test data for SHAP (200 samples for speed)
X_explain = X_test_processed[:200]

# Initialise SHAP explainer
shap_explainer = SHAPExplainer(
    model=model,
    X_data=X_explain,
    feature_names=feature_names,
)

# Compute SHAP values
shap_values = shap_explainer.compute_shap_values()

print(f"SHAP values shape: {shap_values.shape}")
print(f"Explained {shap_values.shape[0]} samples with {shap_values.shape[1]} features")

### 2.2 SHAP Summary Plot (Beeswarm)

This is the most informative SHAP plot — it shows **every feature's impact** on every prediction, with colour indicating feature value (red = high, blue = low).

In [ ]:
# Summary plot (beeswarm)
fig = shap_explainer.plot_summary(save=True)
plt.show()

### 2.3 SHAP Feature Importance Bar Chart

Shows the **mean absolute SHAP value** for each feature — a global ranking of feature importance.

In [ ]:
# Feature importance bar chart
fig = shap_explainer.plot_feature_importance(save=True)
plt.show()

In [ ]:
# Global feature importance as a table
global_importance = shap_explainer.get_global_feature_importance()

importance_df = pd.DataFrame(global_importance)
importance_df.index = range(1, len(importance_df) + 1)
importance_df.index.name = "Rank"

print("\nGlobal Feature Importance (Mean |SHAP Value|)")
print("=" * 50)
display(importance_df.head(15))

### 2.4 SHAP Waterfall Plot — Approved Applicant

The waterfall plot shows how each feature pushes the prediction from the **base value** (average prediction) toward the final output for a **single applicant**.

In [ ]:
# Waterfall plot for approved applicant
print(f"Applicant #{sample_approved + 1} — Predicted: APPROVED (P = {y_proba[sample_approved]:.4f})")

if sample_approved < 200:  # Within our explained subset
    fig = shap_explainer.plot_waterfall(sample_index=sample_approved, save=True)
    plt.show()
else:
    fig = shap_explainer.plot_waterfall(sample_index=0, save=True)
    plt.show()

### 2.5 SHAP Waterfall Plot — Rejected Applicant

In [ ]:
# Waterfall plot for rejected applicant
# Find a rejected applicant within our 200-sample subset
y_pred_subset = model.predict(X_explain)
rejected_in_subset = np.where(y_pred_subset == 0)[0]

if len(rejected_in_subset) > 0:
    rejected_idx = rejected_in_subset[0]
    proba_rejected = model.predict_proba(X_explain[rejected_idx:rejected_idx+1])[:, 1][0]
    print(f"Applicant #{rejected_idx + 1} — Predicted: REJECTED (P = {proba_rejected:.4f})")
    fig = shap_explainer.plot_waterfall(sample_index=rejected_idx, save=True)
    plt.show()
else:
    print("No rejected applicants in subset — using sample index 1")
    fig = shap_explainer.plot_waterfall(sample_index=1, save=True)
    plt.show()

### 2.6 SHAP Force Plot

Force plots provide another view of the same information — features in **red** push the prediction higher (toward approval), features in **blue** push it lower (toward rejection).

In [ ]:
# Force plot (matplotlib version) for an approved applicant
fig = shap_explainer.plot_force(sample_index=0, matplotlib=True, save=True)
plt.show()

In [ ]:
# Force plot (interactive HTML version for notebooks)
if len(rejected_in_subset) > 0:
    force_plot = shap_explainer.plot_force(sample_index=rejected_in_subset[0], matplotlib=False)
    display(force_plot)

### 2.7 Top Contributing Features (Per Prediction)

In [ ]:
# Get top features for the approved applicant
approved_idx_subset = 0  # First sample in the subset
top_features_approved = shap_explainer.get_top_features(approved_idx_subset, top_n=5)

print("\n" + "=" * 60)
print("TOP 5 FEATURES — APPROVED APPLICANT")
print("=" * 60)
for f in top_features_approved:
    icon = "📈" if f["impact"] == "positive" else "📉"
    print(f"  {icon} {f['feature']:35s}  SHAP={f['shap_value']:+.4f}  ({f['impact']})")

# Get top features for the rejected applicant
if len(rejected_in_subset) > 0:
    top_features_rejected = shap_explainer.get_top_features(rejected_in_subset[0], top_n=5)

    print("\n" + "=" * 60)
    print("TOP 5 FEATURES — REJECTED APPLICANT")
    print("=" * 60)
    for f in top_features_rejected:
        icon = "📈" if f["impact"] == "positive" else "📉"
        print(f"  {icon} {f['feature']:35s}  SHAP={f['shap_value']:+.4f}  ({f['impact']})")

### 2.8 SHAP Explanation Object (Native SHAP Plots)

In [ ]:
# Get the full SHAP Explanation object for native SHAP visualisations
explanation_obj = shap_explainer.get_explanation_object()

# Native SHAP beeswarm plot
print("Native SHAP Beeswarm Plot:")
shap.plots.beeswarm(explanation_obj, show=True)

In [ ]:
# Native SHAP bar plot
print("Native SHAP Bar Plot (Global Importance):")
shap.plots.bar(explanation_obj, show=True)

---

## 3. LIME Analysis

**LIME (Local Interpretable Model-agnostic Explanations)** creates a **local linear approximation** around each prediction. It's model-agnostic and explains in human-readable rules.

### 3.1 Initialise LIME Explainer

In [ ]:
from src.explainability.lime_explainer import LIMEExplainer

# Initialise LIME explainer
lime_explainer = LIMEExplainer(
    model=model,
    X_train=X_train_processed,
    feature_names=feature_names,
    class_names=["Rejected", "Approved"],
)

print("✅ LIME explainer initialised")

### 3.2 LIME Explanation — Approved Applicant

In [ ]:
# LIME explanation for an approved applicant
approved_instance = X_test_processed[sample_approved]

print(f"Explaining Approved Applicant (index {sample_approved}, P = {y_proba[sample_approved]:.4f})")

# Get explanation as a structured list
lime_approved = lime_explainer.get_explanation_as_list(approved_instance, num_features=10)

print("\nLIME Feature Contributions (Approved Applicant):")
print("=" * 60)
for e in lime_approved:
    icon = "🟢" if e["impact"] == "positive" else "🔴"
    print(f"  {icon} {e['feature_rule']:45s}  contribution={e['contribution']:+.4f}")

In [ ]:
# LIME explanation plot for approved applicant
fig = lime_explainer.plot_explanation(
    approved_instance,
    num_features=10,
    sample_label="Approved Applicant",
    save=True,
)
plt.show()

### 3.3 LIME Explanation — Rejected Applicant

In [ ]:
# LIME explanation for a rejected applicant
rejected_instance = X_test_processed[sample_rejected]

print(f"Explaining Rejected Applicant (index {sample_rejected}, P = {y_proba[sample_rejected]:.4f})")

lime_rejected = lime_explainer.get_explanation_as_list(rejected_instance, num_features=10)

print("\nLIME Feature Contributions (Rejected Applicant):")
print("=" * 60)
for e in lime_rejected:
    icon = "🟢" if e["impact"] == "positive" else "🔴"
    print(f"  {icon} {e['feature_rule']:45s}  contribution={e['contribution']:+.4f}")

In [ ]:
# LIME explanation plot for rejected applicant
fig = lime_explainer.plot_explanation(
    rejected_instance,
    num_features=10,
    sample_label="Rejected Applicant",
    save=True,
)
plt.show()

### 3.4 What-If Analysis

This is a powerful feature: *"What would happen if the applicant's credit score was higher?"* or *"If their income increased by ₹2L, would they get approved?"*

In [ ]:
# What-If Analysis: Increase credit score for a rejected applicant
# Find the index of 'credit_score' in feature_names
credit_score_idx = None
for i, name in enumerate(feature_names):
    if "credit_score" in name.lower():
        credit_score_idx = i
        break

if credit_score_idx is not None:
    print("What-If Analysis: Changing credit_score")
    print("=" * 60)

    # Try different credit score values
    original_value = rejected_instance[credit_score_idx]
    test_values = [original_value + 0.5, original_value + 1.0, original_value + 1.5, original_value + 2.0]

    print(f"Original standardised value: {original_value:.4f}")
    print()

    for new_val in test_values:
        result = lime_explainer.what_if_analysis(
            rejected_instance,
            feature_index=credit_score_idx,
            new_value=new_val,
        )
        print(
            f"  credit_score: {result['original_value']:.2f} → {result['new_value']:.2f} | "
            f"P(Approved): {result['original_prediction']['probabilities']['Approved']:.4f} → "
            f"{result['modified_prediction']['probabilities']['Approved']:.4f} "
            f"(Δ = {result['probability_change']:+.4f}) | "
            f"Decision: {result['modified_prediction']['class']}"
        )
else:
    print("credit_score feature not found in feature_names")

In [ ]:
# What-If Analysis: Increase income for a rejected applicant
income_idx = None
for i, name in enumerate(feature_names):
    if "person_income" in name.lower() or "income" in name.lower():
        income_idx = i
        break

if income_idx is not None:
    print("What-If Analysis: Changing person_income")
    print("=" * 60)

    original_value = rejected_instance[income_idx]
    test_values = [original_value + 0.5, original_value + 1.0, original_value + 2.0, original_value + 3.0]

    print(f"Original standardised value: {original_value:.4f}")
    print()

    for new_val in test_values:
        result = lime_explainer.what_if_analysis(
            rejected_instance,
            feature_index=income_idx,
            new_value=new_val,
        )
        print(
            f"  income: {result['original_value']:.2f} → {result['new_value']:.2f} | "
            f"P(Approved): {result['original_prediction']['probabilities']['Approved']:.4f} → "
            f"{result['modified_prediction']['probabilities']['Approved']:.4f} "
            f"(Δ = {result['probability_change']:+.4f}) | "
            f"Decision: {result['modified_prediction']['class']}"
        )
else:
    print("income feature not found in feature_names")

In [ ]:
# Visualise What-If: probability curve as credit_score changes
if credit_score_idx is not None:
    score_range = np.linspace(
        rejected_instance[credit_score_idx] - 2,
        rejected_instance[credit_score_idx] + 3,
        50,
    )

    probabilities = []
    for val in score_range:
        modified = rejected_instance.copy()
        modified[credit_score_idx] = val
        prob = model.predict_proba(modified.reshape(1, -1))[0, 1]
        probabilities.append(prob)

    fig, ax = plt.subplots(figsize=(10, 5))
    ax.plot(score_range, probabilities, color="#667eea", linewidth=2.5)
    ax.axhline(y=0.5, color="#e53e3e", linestyle="--", alpha=0.7, label="Decision Threshold (0.5)")
    ax.axvline(
        x=rejected_instance[credit_score_idx],
        color="#38a169", linestyle="--", alpha=0.7,
        label=f"Original Value ({rejected_instance[credit_score_idx]:.2f})",
    )
    ax.fill_between(score_range, probabilities, 0.5, alpha=0.1, color="#667eea")
    ax.set_xlabel("Credit Score (standardised)", fontsize=12)
    ax.set_ylabel("P(Approved)", fontsize=12)
    ax.set_title("What-If Analysis: Impact of Credit Score on Approval Probability",
                 fontsize=14, fontweight="bold")
    ax.legend(fontsize=10)
    ax.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.savefig(REPORTS_DIR / "what_if_credit_score.png", bbox_inches="tight", dpi=150)
    plt.show()
    print(f"Saved → {REPORTS_DIR / 'what_if_credit_score.png'}")

---

## 4. SHAP vs LIME Comparison

Let's compare what SHAP and LIME say about the **same applicant** to see if they agree on which features matter most.

In [ ]:
# Side-by-side comparison for an approved applicant
comparison_idx = 0  # First sample in the SHAP subset
instance = X_explain[comparison_idx]

comparison_result = lime_explainer.explain_and_compare(
    instance=instance,
    shap_explainer=shap_explainer,
    num_features=5,
)

print(f"Prediction: {comparison_result['prediction']}")
print(f"Probabilities: {comparison_result['probability']}")

print("\n" + "=" * 60)
print("SHAP vs LIME — Top Features Comparison")
print("=" * 60)
print(f"{'Feature':<35} {'SHAP Value':>12}  {'LIME Contrib':>14}")
print("-" * 65)

# SHAP top features
print("\nSHAP Top 5:")
for f in comparison_result["shap_explanation"]:
    icon = "📈" if f["impact"] == "positive" else "📉"
    print(f"  {icon} {f['feature']:<35s}  {f['shap_value']:+.4f}")

print("\nLIME Top 5:")
for f in comparison_result["lime_explanation"][:5]:
    icon = "🟢" if f["impact"] == "positive" else "🔴"
    print(f"  {icon} {f['feature_rule']:<35s}  {f['contribution']:+.4f}")

In [ ]:
# Visualise SHAP vs LIME side-by-side
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# SHAP
shap_data = comparison_result["shap_explanation"]
shap_features = [d["feature"] for d in shap_data]
shap_values_plot = [d["shap_value"] for d in shap_data]
shap_colors = ["#38a169" if v > 0 else "#e53e3e" for v in shap_values_plot]

axes[0].barh(shap_features[::-1], shap_values_plot[::-1],
             color=shap_colors[::-1], edgecolor="white")
axes[0].set_title("SHAP Feature Contributions", fontsize=13, fontweight="bold")
axes[0].set_xlabel("SHAP Value")
axes[0].axvline(x=0, color="gray", linestyle="-", alpha=0.5)

# LIME
lime_data = comparison_result["lime_explanation"][:5]
lime_features = [d["feature_rule"] for d in lime_data]
lime_values = [d["contribution"] for d in lime_data]
lime_colors = ["#38a169" if v > 0 else "#e53e3e" for v in lime_values]

axes[1].barh(lime_features[::-1], lime_values[::-1],
             color=lime_colors[::-1], edgecolor="white")
axes[1].set_title("LIME Feature Contributions", fontsize=13, fontweight="bold")
axes[1].set_xlabel("LIME Contribution")
axes[1].axvline(x=0, color="gray", linestyle="-", alpha=0.5)

plt.suptitle(
    f"SHAP vs LIME — Prediction: {comparison_result['prediction']} "
    f"(P = {comparison_result['probability'].get('Approved', 0):.4f})",
    fontsize=15, fontweight="bold",
)
plt.tight_layout()
plt.savefig(REPORTS_DIR / "shap_vs_lime_comparison.png", bbox_inches="tight", dpi=150)
plt.show()
print(f"Saved → {REPORTS_DIR / 'shap_vs_lime_comparison.png'}")

## 5. Save Explainability Summary

In [ ]:
# Save a summary of the explainability analysis
explainability_summary = {
    "model_type": type(model).__name__,
    "shap_method": "TreeExplainer",
    "lime_method": "LimeTabularExplainer",
    "samples_explained": int(X_explain.shape[0]),
    "global_top_5_features": [
        {"feature": f["feature"], "importance": f["importance"]}
        for f in global_importance[:5]
    ],
    "plots_generated": [
        "shap_summary_plot.png",
        "shap_feature_importance.png",
        "shap_waterfall_sample_0.png",
        "shap_force_sample_0.png",
        "lime_explanation_approved_applicant.png",
        "lime_explanation_rejected_applicant.png",
        "what_if_credit_score.png",
        "shap_vs_lime_comparison.png",
    ],
}

with open(REPORTS_DIR / "explainability_summary.json", "w") as f:
    json.dump(explainability_summary, f, indent=2)

print(f"✓ Explainability summary saved → {REPORTS_DIR / 'explainability_summary.json'}")

# List all report files
print(f"\nAll reports in {REPORTS_DIR}:")
for p in sorted(REPORTS_DIR.glob("*")):
    if p.is_file() and p.name != ".gitkeep":
        size_kb = p.stat().st_size / 1024
        print(f"  📄 {p.name:45s} ({size_kb:.1f} KB)")

---

## ✅ Summary

| Analysis | What Was Demonstrated |
|:---------|:---------------------|
| **SHAP Summary Plot** | Global feature impact — shows every feature's effect on every prediction |
| **SHAP Feature Importance** | Mean |SHAP value| ranking — identifies the most important features overall |
| **SHAP Waterfall** | Per-applicant breakdown — shows how each feature pushed the decision |
| **SHAP Force Plot** | Visual push/pull from base value — red (approval) vs blue (rejection) |
| **LIME Explanation** | Local linear approximation — human-readable feature rules |
| **What-If Analysis** | Counterfactual reasoning — "what if credit score was higher?" |
| **SHAP vs LIME** | Side-by-side comparison — both methods generally agree on top features |

### Key Findings

The top features driving loan decisions are:
1. **loan_percent_income** / **loan_to_income_ratio** — how large the loan is relative to income
2. **loan_int_rate** — interest rate (higher = riskier)
3. **credit_score** / **credit_risk_band** — creditworthiness indicator
4. **person_income** — applicant's income level
5. **previous_loan_defaults_on_file** — history of defaults

These align with domain knowledge in financial lending — the model has learned meaningful patterns.

### Regulatory Compliance

This explainability layer satisfies requirements from:
- **RBI Fair Lending Guidelines** — transparent decision-making
- **Equal Credit Opportunity Act** — ability to explain adverse decisions
- **GDPR Article 22** — right to explanation for automated decisions